# P4 electricity — automatic official download

**Upload only this notebook to Colab. No separate ZIP, GitHub checkout or GH_TOKEN is needed.** The notebook contains a checksummed code-only snapshot; data are downloaded from UCI automatically. The complete official archive is cached on Drive; only discovery A columns are parsed as future targets. Reserved B futures are never numerically parsed or used, although they are contained in the downloaded original archive.

Select **T4 GPU**. Run `chronos_2 / smoke`, inspect timing, then `chronos_2 / full`. Next run `timesfm_3 / smoke` and `timesfm_3 / full`, preferably in a fresh GPU session. Finally run `MODE = 'analyze'` (CPU works). Re-running resumes verified units. First use downloads approximately 261 MB and replays the original selection and construct checks.

Same 32 original IDs, three discovery seeds, two mechanisms, frozen checkpoints and P1 thresholds. All four cells are retained; ID-cluster intervals are exploratory, not independent confirmation. No parameter search or primary-result changes. TimesFM-3 is subject to its non-commercial license.

Automatic-version outputs use `MyDrive/tsfm-covariate-faithfulness/p4_electricity_discovery_auto_v1/` to avoid mixing them with the earlier manually uploaded release.


In [ ]:
BACKBONE = 'chronos_2'  # then 'timesfm_3'
MODE = 'smoke'  # smoke -> full for each backbone; then analyze
assert BACKBONE in {'chronos_2', 'timesfm_3'}
assert MODE in {'smoke', 'full', 'analyze'}
print('P4 electricity discovery:', BACKBONE, MODE)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import base64, hashlib, io, json, os, subprocess, sys, zipfile

# Code/config only, not electricity observations or ID-selection receipts.
EXPECTED_SOURCE_SHA256 = 'a7ced42986156c4158bf42162851cd865d171eca74f1a0ab0f9960b66770cb90'
CODE_ZIP_BASE64 = '...'
payload = base64.b64decode(CODE_ZIP_BASE64)
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SOURCE_SHA256, 'Code snapshot checksum failed'
BOOT = Path('/content/p4_auto_bootstrap_' + EXPECTED_SOURCE_SHA256[:12])
BOOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for entry in archive.infolist():
        target = (BOOT / entry.filename).resolve()
        if not target.is_relative_to(BOOT.resolve()):
            raise RuntimeError('Unsafe source snapshot path')
        content = archive.read(entry)
        if target.exists() and target.read_bytes() != content:
            raise RuntimeError('Local bootstrap changed. Use a fresh runtime; do not bypass checks.')
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            target.write_bytes(content)
    expected_py = {n for n in archive.namelist() if n.endswith('.py')}
    actual_py = {p.relative_to(BOOT).as_posix() for p in (BOOT / 'src').rglob('*.py')}
    assert actual_py == expected_py, 'Unexpected stale Python module in bootstrap'
PROJECT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness')
CACHE = PROJECT / 'p4_official_source_cache'
ROOT = PROJECT / ('p4_auto_runtime_' + EXPECTED_SOURCE_SHA256[:12])
OUTPUT = PROJECT / 'p4_electricity_discovery_auto_v1'
print('Code snapshot verified:', EXPECTED_SOURCE_SHA256)
print('Source cache:', CACHE)
print('Durable experiment output:', OUTPUT)

def run_logged(command, *, cwd, env, log_name):
    log_path = OUTPUT / 'logs' / log_name
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as log:
        with subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        raise RuntimeError(f'Stopped with exit code {returncode}. Copy the actual error above. Log: {log_path}')


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(BOOT / 'requirements/colab-base.txt')], check=True)
if MODE != 'analyze':
    subprocess.run([sys.executable, '-c', 'import torch; assert torch.cuda.is_available(), "Select T4 GPU and reconnect"; print(torch.cuda.get_device_name(0), torch.__version__)'], check=True)
bootstrap_env = os.environ.copy()
bootstrap_env['PYTHONPATH'] = str(BOOT / 'src')
run_logged([sys.executable, '-u', '-m', 'covfaith_p4.electricity_autodata',
            '--bootstrap', str(BOOT), '--cache', str(CACHE), '--runtime', str(ROOT)],
           cwd=BOOT, env=bootstrap_env, log_name='automatic_data_setup.log')
manifest = json.loads((ROOT / 'release_manifest.json').read_text())
print('Data/selection/constructs verified. Pilot config:', manifest['config_hash'])


In [ ]:
# Avoid a same-version PyPI TimesFM install shadowing the pinned source package.
if MODE != 'analyze' and BACKBONE == 'timesfm_3':
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'timesfm'], check=True)
if MODE != 'analyze':
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        print('HF_TOKEN unavailable; continuing with public-model access.')
    else:
        os.environ['HF_TOKEN'] = hf_token
        del hf_token
    req = 'chronos2.txt' if BACKBONE == 'chronos_2' else 'timesfm3.txt'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements' / req)], check=True)
    if BACKBONE == 'timesfm_3':
        subprocess.run([sys.executable, '-c', 'from timesfm3 import ModelConfig, TimesFM3Evaluator; print("TimesFM-3 import OK")'], check=True)
    subprocess.run([sys.executable, '-c', 'import torch; assert torch.cuda.is_available(), "Select T4 GPU and reconnect"; print(torch.cuda.get_device_name(0), torch.__version__)'], check=True)
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(ROOT / 'src')
subprocess.run([sys.executable, '-c', 'from pathlib import Path; from covfaith_p4.electricity_pilot import load_release; load_release(Path.cwd()); print("Pilot imports and release checks OK")'], cwd=ROOT, env=ENV, check=True)


In [ ]:
# Fresh subprocess: no stale imports, complete errors streamed and saved.
command = [sys.executable, '-u', '-m', 'covfaith_p4.electricity_pilot',
           '--root', str(ROOT), '--output', str(OUTPUT), '--mode', MODE]
if MODE != 'analyze':
    command += ['--backbone', BACKBONE]
run_logged(command, cwd=ROOT, env=ENV, log_name=f'{BACKBONE}_{MODE}.log')


In [ ]:
if MODE == 'analyze':
    report_path = OUTPUT / 'analysis/discovery_report.json'
    report = json.loads(report_path.read_text())
    print(json.dumps(report, indent=2))
    print('Send discovery_report.json and complete_four_cell_matrix.csv, not private NPZ arrays.')
else:
    report_path = OUTPUT / MODE / BACKBONE / 'completion.json'
    report = json.loads(report_path.read_text())
    print(json.dumps(report, indent=2))
    if MODE == 'smoke':
        estimate = report['linear_full_time_projection_seconds_not_guarantee'] / 60
        print(f'Linear full-run projection: {estimate:.1f} minutes for this backbone (not a guarantee; batching and warm-up differ).')
        print('If timing is acceptable, set MODE = full and rerun. Retain this fixed configuration.')
    else:
        print('Run the other backbone smoke/full next. After both complete, set MODE = analyze.')
print('Private report:', report_path)
